In [1]:
from time import time
from uuid import uuid4

PROTOCOL_VERSION = "1.0"

def create_user_message_request(
    message: str,
    session_id: str,
    *,
    scene_id: str | None = None,
    scene_revision: int = 0,
    scene_summary: dict | None = None,
    selection: list[str] | None = None,
    blueprint: dict | None = None,
    thinking_mode: bool = False,
    precision_mode: bool = False,
    procedural_materials_enabled: bool = False,
    plan_mode: bool = False,
    recent_messages: list[dict] | None = None,
) -> dict:
    return {
        "type": "user_message",
        "protocol_version": PROTOCOL_VERSION,
        "request_id": f"req_{int(time() * 1000)}_{uuid4().hex[:9]}",
        "session_id": session_id,
        "scene_id": scene_id,
        "scene_revision": scene_revision,
        "message": message,
        "scene_summary": scene_summary or {},
        "selection": list(selection or []),
        "blueprint": blueprint,
        "thinking_mode": thinking_mode,
        "precision_mode": precision_mode,
        "procedural_materials_enabled": procedural_materials_enabled,
        "plan_mode": plan_mode,
        "recent_messages": list(recent_messages or []),
    }

request = create_user_message_request(
    "生成一个小型住宅",
    "session-demo",
    scene_id="scene-1",
    scene_revision=3,
    scene_summary={"element_count": 0},
    selection=[],
    blueprint={"version": "1.0", "elements": []},
    plan_mode=True,
)
request

{'type': 'user_message',
 'protocol_version': '1.0',
 'request_id': 'req_1788868124536_9b569ec7e',
 'session_id': 'session-demo',
 'scene_id': 'scene-1',
 'scene_revision': 3,
 'message': '生成一个小型住宅',
 'scene_summary': {'element_count': 0},
 'selection': [],
 'blueprint': {'version': '1.0', 'elements': []},
 'thinking_mode': False,
 'precision_mode': False,
 'procedural_materials_enabled': False,
 'plan_mode': True,
 'recent_messages': []}

In [2]:
import json

wire_text = json.dumps(request, ensure_ascii=False)
decoded = json.loads(wire_text)

print(type(request).__name__, "→", type(wire_text).__name__, "→", type(decoded).__name__)
print(wire_text)
assert decoded == request

dict → str → dict
{"type": "user_message", "protocol_version": "1.0", "request_id": "req_1788868124536_9b569ec7e", "session_id": "session-demo", "scene_id": "scene-1", "scene_revision": 3, "message": "生成一个小型住宅", "scene_summary": {"element_count": 0}, "selection": [], "blueprint": {"version": "1.0", "elements": []}, "thinking_mode": false, "precision_mode": false, "procedural_materials_enabled": false, "plan_mode": true, "recent_messages": []}


In [3]:
request_a = create_user_message_request("第一条", "session-demo")
request_b = create_user_message_request("第二条", "session-demo")

assert request_a["session_id"] == request_b["session_id"]
assert request_a["request_id"] != request_b["request_id"]
print("同一会话中的两个独立请求")

同一会话中的两个独立请求


In [4]:
def validate_user_message(data: dict) -> list[str]:
    errors = []
    if data.get("type") != "user_message":
        errors.append("type 必须是 user_message")
    if not str(data.get("request_id", "")).strip():
        errors.append("缺少 request_id")
    if not str(data.get("message", "")).strip():
        errors.append("message 不能为空")
    return errors

cases = [
    (request, []),
    ({"type": "user_message", "message": "你好"}, ["缺少 request_id"]),
    ({"type": "ping", "request_id": "r1", "message": "你好"}, ["type 必须是 user_message"]),
]

for payload, expected in cases:
    actual = validate_user_message(payload)
    print(actual)
    assert actual == expected

[]
['缺少 request_id']
['type 必须是 user_message']
